#Set up

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Função do gráfico

In [ ]:
def grafico_dinamica_entrada_saida(
    turnover_plot,
    detalhe="completo",
    titulo="Dinâmica mensal de beneficiários – Entradas e Saídas"
):
    # Cria uma cópia do DataFrame para evitar modificar o original
    # e garante que o índice esteja sequencial
    df = turnover_plot.copy().reset_index(drop=True)

    # Define os valores permitidos para o parâmetro 'detalhe'
    detalhes_validos = {"completo", "pop", "yoy", "nenhum"}

    # Validação do parâmetro 'detalhe'
    if detalhe not in detalhes_validos:
        raise ValueError(
            f"Parâmetro 'detalhe' inválido. Use um destes: {sorted(detalhes_validos)}"
        )

    # =========================
    # 1) Métricas auxiliares
    # =========================

    # Calcula variação percentual mês a mês (PoP - Period over Period)
    df["pop_entradas"] = df["entradas"].pct_change() * 100
    df["pop_saidas"] = df["saidas"].pct_change() * 100

    # Calcula variação percentual ano a ano (YoY - Year over Year)
    # Considera defasagem de 12 períodos (meses)
    df["yoy_entradas"] = df["entradas"].pct_change(periods=12) * 100
    df["yoy_saidas"] = df["saidas"].pct_change(periods=12) * 100

    # Calcula a média das entradas e saídas
    media_entradas = df["entradas"].mean()
    media_saidas = df["saidas"].mean()

    # Cria versões formatadas (texto) para exibir dentro das barras
    # Substitui vírgula por ponto para padrão brasileiro
    df["texto_entradas"] = df["entradas"].apply(lambda v: f"{v:,.0f}".replace(",", "."))
    df["texto_saidas"] = df["saidas"].apply(lambda v: f"-{v:,.0f}".replace(",", "."))

    # Inicializa a figura do Plotly
    fig = go.Figure()

    # =========================
    # 2) Barras - Entradas
    # =========================

    # Adiciona barras positivas (entradas)
    fig.add_trace(
        go.Bar(
            x=df["competencia"],             # eixo x (tempo)
            y=df["entradas"],                # valores positivos
            name="Entradas",
            marker_color="#2ca02c",          # verde
            text=df["texto_entradas"],       # texto dentro da barra
            textposition="inside",           # posição interna
            textfont=dict(color="white", size=11),
            insidetextanchor="middle"
        )
    )

    # =========================
    # 3) Barras - Saídas
    # =========================

    # Adiciona barras negativas (saídas)
    # Multiplica por -1 para aparecer abaixo do eixo zero
    fig.add_trace(
        go.Bar(
            x=df["competencia"],
            y=-df["saidas"],                # negativo para inverter direção
            name="Saídas",
            marker_color="#d62728",         # vermelho
            text=df["texto_saidas"],
            textposition="inside",
            textfont=dict(color="white", size=11),
            insidetextanchor="middle"
        )
    )

    # =========================
    # 4) Anotações - Entradas
    # =========================

    # Só adiciona anotações se o detalhe não for "nenhum"
    if detalhe != "nenhum":
        for _, row in df.iterrows():
            textos = []

            # Adiciona PoP se configurado
            if detalhe in {"completo", "pop"} and pd.notna(row["pop_entradas"]):
                seta = "▲" if row["pop_entradas"] > 0 else "▼"
                textos.append(f"PoP {seta} {abs(row['pop_entradas']):.0f}%")

            # Adiciona YoY se configurado
            if detalhe in {"completo", "yoy"} and pd.notna(row["yoy_entradas"]):
                seta = "▲" if row["yoy_entradas"] > 0 else "▼"
                textos.append(f"YoY {seta} {abs(row['yoy_entradas']):.0f}%")

            # Se houver textos, adiciona anotação acima da barra
            if textos:
                fig.add_annotation(
                    x=row["competencia"],
                    y=row["entradas"],
                    text="<br>".join(textos),   # quebra de linha entre métricas
                    showarrow=False,
                    font=dict(size=10, color="green"),
                    yshift=28                  # deslocamento vertical acima da barra
                )

    # =========================
    # 5) Anotações - Saídas
    # =========================

    if detalhe != "nenhum":
        for _, row in df.iterrows():
            textos = []

            # PoP para saídas
            if detalhe in {"completo", "pop"} and pd.notna(row["pop_saidas"]):
                seta = "▲" if row["pop_saidas"] > 0 else "▼"
                textos.append(f"PoP {seta} {abs(row['pop_saidas']):.0f}%")

            # YoY para saídas
            if detalhe in {"completo", "yoy"} and pd.notna(row["yoy_saidas"]):
                seta = "▲" if row["yoy_saidas"] > 0 else "▼"
                textos.append(f"YoY {seta} {abs(row['yoy_saidas']):.0f}%")

            # Anotação abaixo da barra
            if textos:
                fig.add_annotation(
                    x=row["competencia"],
                    y=-row["saidas"],
                    text="<br>".join(textos),
                    showarrow=False,
                    font=dict(size=10, color="red"),
                    yshift=-28                # deslocamento para baixo
                )

    # =========================
    # 6) Linhas médias
    # =========================

    # Linha horizontal da média de entradas
    fig.add_hline(
        y=media_entradas,
        line_dash="dash",
        line_color="#2ca02c"
    )

    # Linha horizontal da média de saídas (negativa)
    fig.add_hline(
        y=-media_saidas,
        line_dash="dash",
        line_color="#d62728"
    )

    # Anotação da média de entradas (lado direito do gráfico)
    fig.add_annotation(
        x=1.02,                         # fora do eixo x (lado direito)
        xref="paper",
        y=media_entradas,
        yref="y",
        text=f"Média: {media_entradas:,.0f}".replace(",", "X").replace(".", ",").replace("X", "."),
        showarrow=False,
        font=dict(size=11, color="#2ca02c"),
        align="left",
        xanchor="left",
        yanchor="middle",
        bgcolor="rgba(255,255,255,0.8)"
    )

    # Anotação da média de saídas
    fig.add_annotation(
        x=1.02,
        xref="paper",
        y=-media_saidas,
        yref="y",
        text=f"Média: -{media_saidas:,.0f}".replace(",", "X").replace(".", ",").replace("X", "."),
        showarrow=False,
        font=dict(size=11, color="#d62728"),
        align="left",
        xanchor="left",
        yanchor="middle",
        bgcolor="rgba(255,255,255,0.8)"
    )

    # =========================
    # 7) Layout final
    # =========================

    fig.update_layout(
        title=titulo,
        barmode="relative",  # permite barras positivas e negativas no mesmo eixo
        xaxis=dict(title="Competência"),
        yaxis=dict(
            title="Entradas / Saídas",
            zeroline=True,           # linha do zero destacada
            zerolinewidth=2,
            zerolinecolor="gray"
        ),
        legend=dict(
            orientation="h",         # legenda horizontal
            yanchor="bottom",
            y=1.05,
            xanchor="right",
            x=1
        ),
        template="plotly_white",
        width=2400,
        height=800,
        margin=dict(l=40, r=140, t=80, b=80)
    )

    return fig

# Dados de demostração

In [ ]:
def gerar_dados_teste(n_periodos=24, seed=42):
    np.random.seed(seed)

    datas = pd.date_range(start="2023-01-01", periods=n_periodos, freq="M")

    entradas = np.random.randint(800, 1500, size=n_periodos)
    saidas = np.random.randint(700, 1400, size=n_periodos)

    df = pd.DataFrame({
        "competencia": datas.strftime("%Y-%m"),
        "entradas": entradas,
        "saidas": saidas
    })

    return df

df_teste = gerar_dados_teste()

/tmp/ipykernel_3748/3468745546.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  datas = pd.date_range(start="2023-01-01", periods=n_periodos, freq="M")


In [ ]:
df_teste.head(3)

,competencia,entradas,saidas
0,2023-01,902,976
1,2023-02,1235,860
2,2023-03,1070,1159


# Plotagem

In [ ]:
fig = grafico_dinamica_entrada_saida(df_teste, detalhe="nenhum")
fig.show()